#### Notebook GeoTiff to Zarr

Goal: Show how a GeoTiff is converted to a Zarr store

Steps:
- download/access the data
- data conversion to Zarr
- compare data access (partial reads) ? not implemented yet -> speed or code differences
- also add virtualizarr

In [2]:
import xarray as xr
import rioxarray
import pystac_client
import requests
import rasterio
import time
import zarr
from geozarr_toolkit import detect_conventions, validate_group
from tqdm import tqdm
from pathlib import Path
from typing import List, Dict
import pickle

import datetime
from geozarr_toolkit import (
    ProjConventionMetadata,
    SpatialConventionMetadata,
    create_proj_attrs,
    create_spatial_attrs,
    create_zarr_conventions,
)

import numpy as np
import pandas as pd

In [3]:
datapath = Path('../data/02_CGLS_SSM_geotif')
CGLS_path = Path('../outputs/02_CGLS_SSM_zarr')

datapath.mkdir(parents=True, exist_ok=True)
CGLS_path.mkdir(parents=True, exist_ok=True)

In [4]:
# read in and download 500 stac items for bbox and downlaod

def download_from_STAC(collection: str, 
                       bbox: List[float],
                       daterange: str,
                       outpath: Path,
                       num_items: int | None = None,
                       download: bool = True) -> Dict[str, datetime.datetime]:
    
    stac_url = "https://stac.eodc.eu/api/v1/"
    eodc = pystac_client.Client.open(stac_url)
    
    found = eodc.search(
        collections=[collection],
        bbox=bbox,
        datetime=daterange,
        max_items=num_items
    )

    meta = {}

    for item in tqdm(found.items()):
        href = item.assets['SSM'].href
        filename = outpath / f"{item.id}.tif"

        meta[filename.name] = item.datetime

        if download and not filename.exists():
            response = requests.get(href, stream=True)
            with open(filename, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)

    return meta

# download the images into local storage, also add temporal metadata information into a pickle if the kernel crashes
time_metadata = download_from_STAC(collection='CGLS_SSM_1KM',
                   bbox=[7.,46.,10.,50.],
                   daterange="2018-07-01/2026-07-31",
                   outpath=datapath,
                   num_items=500,
                   download=True)

with open(datapath / 'metadata.pickle', 'wb') as file:
    pickle.dump(time_metadata, file)

372it [00:49,  7.53it/s]


In [5]:
# Create an xarray datacube stacking all the .tif files along the temporal dimension -> the foundation of writing to a zarr store
datasets = []
for fn in datapath.glob('*.tif'):
    # as we want to write the data to an optimised file format, we will not apply masking and scaling to save storage/conversion space later on
    da = rioxarray.open_rasterio(fn, mask_and_scale=False)
    
    # concat wouldnt like different crs, and while they are all in the same equigrid, its still relevant to check
    if da[0].rio.crs.to_epsg() == 27704:
        da = da[0].expand_dims(time=[time_metadata[fn.name].replace(tzinfo=None)])
        datasets.append(da)
    else:
        print(f'different crs detected, skipping file: {fn}')

# merge the datasets along the time dimension
cube = xr.concat(datasets, dim="time", fill_value=255)

/tmp/ipykernel_56054/1389583910.py:15: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'x' ('x',) The recommendation is to set join explicitly for this case.
  cube = xr.concat(datasets, dim="time", fill_value=255)
/tmp/ipykernel_56054/1389583910.py:15: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'y' ('y',) The recommendation is to set join explicitly for this case.
  cube = xr.concat(datasets, dim="time", fill_value=255)


Once we have created the datacube, we can write to a file. Writing it to a COG is not (really) possible, as too many bands can overwhealm the GDAL processes behind rasterio. Theoretically, GDAL allows rasters with up to 65 536 bands, but the sample below already lead to an OOM error. Thus, users are limited in writing multidimensional data to a COG, whereas the Zarr format allows it. Below are samples for aggregating and writing timeseries data into COGs.

In [6]:
# Testing out a write to a GeoTiff: selecting timeseries from smaller spatial subset to create mean over whole image set
subcube = cube.sel(y=slice(1840500, 1850500), x=slice(4970500, 4980500))
ts = subcube.mean(dim=["time"], skipna=True)
ts.rio.to_raster(
        CGLS_path / 'subcube_mean_over_time.tif',
        compress="ZSTD",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        driver='COG',
        dtype=ts.dtype, # set explicitly if required by data type -> eg S1 
    )

# select the first five time slices -> will be written into geotiff bands
# this works here as 3 dimensional data is allowed in geotiff (time, x, y) but 4 dimensional would throw an error -> (time, band (r, g, b, nir), x, y)
subcube = cube.isel(time=slice(0, 5))
subcube.rio.to_raster(
        CGLS_path / 'cube_time_0_to_5.tif',
        compress="ZSTD",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        driver='COG',
        dtype=cube.dtype, # set explicitly if required by data type -> eg S1 
    )

# also, writing many bands to geotiff will use up large amounts of memory, as time*x*y initialises large arrays -> pixel compression or band compressio more efficient?
test_oom_geotif_writing = False
if test_oom_geotif_writing:
    # will fail into kernel dying
    cube.rio.to_raster(
            CGLS_path / 'test_cube_full_time.tif',
            compress="ZSTD",
            tiled=True,
            blockxsize=512,
            blockysize=512,
            driver='COG',
            dtype=cube.dtype, 
        )

There are several options for writing zarr stores, and as it is relatively newly adopted by the geospatial community the ecosystem and its specifications are still evolving. First, a method for writing a zarr store with xarray is shown, which does not (automatically) implement the geozarr specification. While geospatial attributes are added (eg. spatial_ref), its not aligned to the specification. Writing with the geozarr specification becomes possible when adding the metadata by writing the zarr store with the native zarr library. It was not investigated, if adding more metadata to the xarray allows geozarr specification adherence. Another option would be to initialise the empty zarr store with geozarr metadata using the native zarr library, closing it, reopening and filling it with xarray, but this was not evaluated.


In [7]:
# easiest implementation using xarray:
# does NOT automatically implements geozarr!
cube_zarr = xr.Dataset({'values': cube.rio.write_nodata(255)})

# add cf metadata. all of it necessary?
cube_zarr = cube_zarr.rio.write_crs(cube.rio.crs).rio.write_coordinate_system().rio.write_transform()

zarr_outpath = CGLS_path / 'xarray_cube.zarr'

# define the chunking along time x width x height
encoding = {'values': {"chunks": (16, 256, 256), "dtype": 'uint8'}}

cube_zarr.to_zarr(zarr_outpath, mode='w', zarr_format=3, compute=False, encoding=encoding)
zarr.consolidate_metadata(zarr_outpath)

# -> other option
# initialise empty zarr store woth cf metadata 
# reopen with xarray and fill with data

/home/samuel/coding/cn_datacube_notebooks/.venv/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


<Group file://../outputs/02_CGLS_SSM_zarr/xarray_cube.zarr>

In [8]:
# native zarr implementation
# need to write geozarr metadata ourselfs: https://developmentseed.org/geozarr-examples/examples/cog-to-zarr/#step-2-build-convention-metadata-from-the-cog
native_zarr_outpath = CGLS_path / 'zarr_cube.zarr'

root = zarr.open_group(
    native_zarr_outpath,
    mode="w",
    zarr_format=3,
)

# metadata -> no multiscales/overviews are generated here as the low spatial resolution doesnt require multiscales
geozarr_attrs = create_proj_attrs(code=f"EPSG:{cube.rio.crs.to_epsg()}")
geozarr_attrs.update(
    create_spatial_attrs(
        dimensions=["y", "x"],
        bbox=list(cube.rio.bounds()),
    )
)
geozarr_attrs["zarr_conventions"] = create_zarr_conventions(
    ProjConventionMetadata(),
    SpatialConventionMetadata(),
)
root.attrs.update(geozarr_attrs)

arr = root.create_array(
    "x",
    shape=cube.x.shape,
    dtype=cube.x.dtype,
    chunks=(256,),
    dimension_names=("x",),
)
root["x"][:] = cube.x.values

root.create_array(
    "y",
    shape=cube.y.shape,
    dtype=cube.y.dtype,
    chunks=(256,),
    dimension_names=("y",),
)
root["y"][:] = cube.y.values

root.create_array(
    "time",
    shape=cube.time.shape,
    dtype=cube.time.dtype,
    chunks=(16,),
    dimension_names=("time",),
)
root["time"][:] = cube.time.values

root.create_array(
    "values",
    shape=cube.shape,
    dtype=cube.dtype,
    chunks=(16, 256, 256),
    fill_value=255,
    dimension_names=("time", "y", "x"),
)
root["values"][:] = cube.values

zarr.consolidate_metadata(native_zarr_outpath)

<Group file://../outputs/02_CGLS_SSM_zarr/zarr_cube.zarr>

In [9]:
def validate_zarr(zarr_path: Path):
    print(f'validating {zarr_path}')
    root = zarr.open_group(zarr_path, mode="r")

    detected = detect_conventions(dict(root.attrs))
    print(f"Detected conventions: {detected}")

    results = validate_group(root)
    if results:
        for conv, errors in results.items():
            status = "PASS" if not errors else "FAIL"
            print(f"  [{status}] {conv}")
            for err in errors:
                print(f"         {err}")

        print(f"\nStore tree:")
        print(root.tree())
    else:
        print(f'store {zarr_path} not conform')

    # not valid -> cube is referenced but not assigned
    print(zarr_path)
    ref_cube = xr.open_dataset(zarr_path)
    return ref_cube.sel(y=slice(1400500, 1500500), x=slice(5070500, 5200500)).mean(dim=["time"], skipna=True)


# Evaluation of geozarr conformity on the two zarr stores
cube_native = validate_zarr(native_zarr_outpath)
cube_xr = validate_zarr(zarr_outpath)
arrs_same = bool((cube_native == cube_xr).all())

print(f'Arrays were loaded similarily: {arrs_same}')

validating ../outputs/02_CGLS_SSM_zarr/zarr_cube.zarr
Detected conventions: ['spatial', 'proj']
  [PASS] spatial
  [PASS] proj
  [PASS] zarr_conventions

Store tree:
/
├── time (372,) datetime64[us]
├── values (372, 1200, 1200) uint8
├── x (1200,) float64
└── y (1200,) float64

../outputs/02_CGLS_SSM_zarr/zarr_cube.zarr
validating ../outputs/02_CGLS_SSM_zarr/xarray_cube.zarr
Detected conventions: []
store ../outputs/02_CGLS_SSM_zarr/xarray_cube.zarr not conform
../outputs/02_CGLS_SSM_zarr/xarray_cube.zarr
Arrays were loaded similarily: True


#### Virtualisation

Instead of reprocessing (which consumes resources and may not be necessary for all applications) we can also virtualize the data repository. Virtualizing works by saving only the metadata of files into a zarr reable array (made possible by Icechuk or Kerchunk) with the only limitation being the original size of input files *coordinates*, as these will have to be loaded into memory when loading the data. Virtualising 1GB files will not make sense, as the whole file coordinates will have to be read into memory when accessing. In such a case reprocessing (to zarr or COG) makes more sense. On the other hand, virtualising a large number of smalle images with large temporal dimesions makes sense, as this allows us to read timeseries data quickly and with familiar tools using the xarray.

In [10]:
# virtualising dependant packages -> code can change for different versions of virtualizarr and Icechunk
import icechunk
import warnings

from virtualizarr import open_virtual_dataset
from virtual_tiff import VirtualTIFF
from obstore.store import LocalStore
from obspec_utils.registry import ObjectStoreRegistry
from zarr.errors import ZarrUserWarning
from imagecodecs.zarr import register_codecs
register_codecs()

warnings.filterwarnings(
    "ignore",
    message="Imagecodecs codecs are not in the Zarr version 3 specification*.",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Codec 'imagecodecs_lzw' not configured in config. Selecting any implementation.",
    category=ZarrUserWarning,
)

In this sample the downloaded SM data will not be aggregated into a single datacube, as concatonating the image coordinates into a single array would require loading the full arrays into memory and processing them to align their respective spatial positioning. Instead we will structure them according to the Equi7Grids-tiling scheme (in which the data is already published). This reduces processign requirements and allows the virtualisation process. If a single output store is wanted, the data has to be reprocessed as shown earlier such a pipeline is currently not supported by the virtualizarr API -> https://github.com/virtual-zarr/virtual-tiff/issues/55

In [11]:
# virtualise zarr: https://icechunk.io/en/latest/guides/virtual/#creating-a-virtual-dataset-with-virtualizarr

# define filepaths for reading in metadata
root = datapath.resolve()
store = LocalStore(prefix=root)
registry = ObjectStoreRegistry({f"file://{root}": store})

# load intermediary metadata for timestamps of tif files
fp = datapath / 'metadata.pickle'
with open(fp, 'rb') as file:
    metadata = pickle.load(file)

# read in tile metadata for tile-dependant processing
tiles = {}
for fn in root.glob("*.tif"):
    # get tile id from input filename
    tile_id = str(fn.stem).split('_')[-1][:-2]
    if tile_id in tiles.keys():
        tiles[tile_id].append(fn)
    else:
        tiles[tile_id] = [fn]

In [15]:
# actually grab the metadata from each iamge and save it to virtualise the input
for tile_id, fns in tiles.items():
    datasets = {}
   
    virtualzarr_outpath = CGLS_path / f'virtual_tile_{tile_id}.icechunk'

    if virtualzarr_outpath.exists():
        print(f'{virtualzarr_outpath} already exists, skipping this iteration -> icechunk writing history doesnt allow reinitialisation')
    else:
        # gather the metadata of each image
        for fn in tqdm(fns, desc=f'Processing: {tile_id}'):
            url = fn.resolve().as_uri()
        
            # load array -> not into memory except the metadata
            ds = open_virtual_dataset(
                url=url,
                registry=registry,
                parser=VirtualTIFF(ifd=0),
                loadable_variables=['x', 'y']
            )

            # select only the first overview data variable to unpack the ds
            ds = ds.rename({'0': 'values'})
            ds = ds.expand_dims(time=[metadata[fn.name].replace(tzinfo=None)])

            # no spatial metadata present -> add coordinates 
            with rasterio.open(fn, 'r') as src:
                transform = src.transform
                width = src.width
                height = src.height

                x = transform.c + (np.arange(width) + 0.5) * transform.a
                y = transform.f + (np.arange(height) + 0.5) * transform.e

                ds = ds.assign_coords(
                    x=("x", x),
                    y=("y", y),
                )
                ds = ds.rio.write_crs(src.crs)
            
            datasets[fn] = ds

        # concating across tiles is not possible as it Reequires fancz indexing -> Rearrange by tiles 
        virtual_cube = xr.concat(datasets.values(), 
                        dim="time", 
                        fill_value=255, 
                        join='outer')
        
        # writing to icechunk
        # directory containing the original tif files
        config = icechunk.RepositoryConfig.default()
        tiff_storage = icechunk.ObjectStoreConfig.LocalFileSystem(str(root))
        config.set_virtual_chunk_container(icechunk.VirtualChunkContainer(url_prefix=f"{root.as_uri()}/", 
                                                                        store=tiff_storage))
        
        # define directory for output directory
        storage = icechunk.local_filesystem_storage(str(virtualzarr_outpath))
        repo = icechunk.Repository.create(storage=storage,
                                        config=config)
        session = repo.writable_session('main')
        
        # actually write data
        virtual_cube.vz.to_icechunk(session.store)
        snapshot = session.commit("Initial virtual cube")

Processing: E042N012: 100%|██████████| 93/93 [00:01<00:00, 74.34it/s]
  2026-07-15T13:08:48.872498Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324

Processing: E048N018: 100%|██████████| 92/92 [00:01<00:00, 78.70it/s]
  2026-07-15T13:08:50.092992Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324

Processing: E042N018: 100%|██████████| 93/93 [00:01<00:00, 82.45it/s]
  2026-07-15T13:08:51.273053Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores

In [16]:
# Opening a Virtual Store

repo_path = CGLS_path / 'virtual_tile_E048N012.icechunk'
storage = icechunk.local_filesystem_storage(str(repo_path))
repo = icechunk.Repository.open(storage=storage)

session = repo.readonly_session("main")
store = session.store
virtual_ds = xr.open_zarr(store)

  2026-07-15T13:08:56.941209Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324



In [17]:
# Validate the files can be accessed similarily and return the same content
ref_df = xr.open_dataset(native_zarr_outpath)
ref_cube = ref_df.sel(y=slice(1400500, 1500500), x=slice(5070500, 5200500)).mean(dim=["time"], skipna=True)

virtual_ds.sel(y=slice(1400500, 1500500), x=slice(5070500, 5200500)).mean(dim=["time"], skipna=True)
arrs_same = bool((ref_cube == virtual_ds).all())

print(f'Arrays were loaded similarily: {arrs_same}')

Arrays were loaded similarily: True
